In [0]:
import sys
sys.path.append("/Workspace/Shared")
import aws_utils
from pyspark.sql import functions as F

env = aws_utils.get_env_prefix()

TARGET_TABLE = f"`{env}-cdh-delivery`.it.ref_paymentplan"

business_date_param = dbutils.widgets.get("run_date").strip()

if business_date_param:
    base_date = business_date_param
else:
    base_date = str(
        spark.sql(f"SELECT MAX(BusinessDate) AS bd FROM `{env}-cdh`.primaryactive.exposure")
        .collect()[0].bd
    )


In [0]:
# Step 1: Build pool - UNION of subsidy sources

df_app_subsidy = (
    spark.table(f"`{env}-cdh`.landed.rpt_applicationsubsidy")
    .filter(
        (F.col("new_account_number").isNotNull()) &
        (F.col("CurrentRecord") == 1) &
        (F.col("PlatformName").isNotNull())
    )
    .select(
        F.col("PlatformName").alias("platformName"),
        F.col("ApplicationNumber").alias("applicationNumber"),
        F.col("DebtNumber").alias("debtNumber"),
        F.col("new_account_number").alias("AccountNumber"),
        F.lit(None).cast("string").alias("settlementCode")
    )
)

df_subsidy_hd = (
    spark.table(f"`{env}-cdh`.landed.profits_lns_ext_subsidy_hd")
    .select(
        F.when(F.col("DEBT_TYPE") == "3", "bridge2")
         .when(F.col("DEBT_TYPE") == "1", "bridge1")
         .when(F.col("DEBT_TYPE") == "2", "N4605")
         .when(F.col("DEBT_TYPE") == "4", "bridge3")
         .alias("platformName"),
        F.col("APPLICATION_NO").alias("applicationNumber"),
        F.col("DEBT_NUMBER").alias("debtNumber"),
        F.col("ACCOUNT_NUMBER").alias("AccountNumber"),
        F.lit(None).cast("string").alias("settlementCode")
    )
)

df_subsidy_keyd = (
    spark.table(f"`{env}-cdh`.landed.rpt_applicationsubsidy")
    .filter(F.col("PlatformName") == "Bridge3")
    .select(
        F.lit("bridge3").alias("platformName"),
        F.col("ApplicationNumber").alias("applicationNumber"),
        F.col("DebtNumber").alias("debtNumber"),
        F.col("new_account_number").alias("AccountNumber"),
        F.lit(None).cast("string").alias("settlementCode")
    )
)

df_pool = df_app_subsidy.unionByName(df_subsidy_hd).unionByName(df_subsidy_keyd)


In [0]:
# Step 2: Join pool with PaymentSchedule

df_distinct_accounts = df_pool.select("AccountNumber").distinct()

df_payment_schedule = (
    spark.table(f"`{env}-cdh`.primaryactive.paymentschedule")
    .filter(
        (F.col("BusinessDate") == F.lit(base_date)) &
        (F.col("SourceSystem") == "profits")
    )
)

df_result = (
    df_distinct_accounts
    .join(df_payment_schedule, df_distinct_accounts["AccountNumber"] == df_payment_schedule["ExposureKey"], "inner")
    .select(
        df_distinct_accounts["AccountNumber"].alias("AssetKey"),
        df_payment_schedule["ExposureKey"].alias("loanid"),
        df_payment_schedule["InstallmentNumber"],
        df_payment_schedule["InstallmentDate"],
        df_payment_schedule["InstallmentAmount"],
        df_payment_schedule["InstallmentTypeCode"].alias("requestType"),
        df_payment_schedule["RequestSN"],
        df_payment_schedule["InstallmentStatusCode"].alias("status"),
        df_payment_schedule["InterestAmount"],
        df_payment_schedule["CapitalAmount"]
    )
)


In [0]:
# Step 3: Write to target table (replaceWhere by base_date)

df_result = df_result.withColumn("base_date", F.lit(base_date).cast("date"))

(
    df_result.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"base_date = '{base_date}'")
    .saveAsTable(TARGET_TABLE)
)